# Spheroid Sweep HDF5 Reader

DDA spheroid parameter sweep の結果ファイルを読み出すノートブック。

- **Cell 1**: ファイルを開き、全パラメータ軸の一覧・範囲・格子点数・波長グループを表示
- **Cell 2**: パラメータ値またはインデックスを指定して結果を取得

In [1]:
"""Cell 1: Open HDF5 file and display parameter overview."""

import h5py
import numpy as np

H5_PATH = "dda_results/dda_results_spheroid_sweep_20260329.h5"

f = h5py.File(H5_PATH, "r")

# ── Wavelength groups ─────────────────────────────────────────────
wl_groups = {}
for name in sorted(f):
    obj = f[name]
    if isinstance(obj, h5py.Group) and "wl_0" in obj.attrs:
        wl_groups[name] = obj.attrs["wl_0"]

print("=" * 60)
print("Wavelength groups")
print("=" * 60)
for name, wl in wl_groups.items():
    m_m = f[name].attrs.get("m_m", "N/A")
    print(f"  {name:12s}  wl_0 = {wl:.4f} um,  m_m = {m_m}")

# ── Parameter grids ───────────────────────────────────────────────
grid_keys = [
    ("D_ve_grid",              "D_ve [um]"),
    ("RI_real_grid",           "Re(m_p)"),
    ("log_AR_grid",            "log10(AR)"),
    ("cos_theta_o_half_grid",  "cos(theta_o) half-domain"),
    ("phi_o_grid",             "phi_o [rad]"),
]

print("\n" + "=" * 60)
print("Parameter grids  (axis order in data arrays)")
print("=" * 60)
for i, (key, label) in enumerate(grid_keys):
    v = f[key][:]
    print(f"  axis {i}: {label:30s}  N={len(v):3d}   "
          f"[{v[0]:.4f} .. {v[-1]:.4f}]   step≈{np.mean(np.diff(v)):.4f}")

# ── Data variables per wavelength group ────────────────────────────
sample_grp_name = list(wl_groups.keys())[0]
sample_grp = f[sample_grp_name]
print("\n" + "=" * 60)
print(f"Datasets in each wavelength group (example: {sample_grp_name})")
print("=" * 60)
for dk in sorted(sample_grp):
    ds = sample_grp[dk]
    print(f"  {dk:25s}  shape={str(ds.shape):30s}  dtype={ds.dtype}")

print("\nFile kept open as `f`.  Close with `f.close()` when done.")

Wavelength groups
  wl_0p453      wl_0 = 0.4530 um,  m_m = 1.0
  wl_0p638      wl_0 = 0.6380 um,  m_m = 1.0
  wl_0p834      wl_0 = 0.8340 um,  m_m = 1.0

Parameter grids  (axis order in data arrays)
  axis 0: D_ve [um]                       N= 16   [0.2600 .. 0.5600]   step≈0.0200
  axis 1: Re(m_p)                         N= 21   [1.2000 .. 1.7000]   step≈0.0250
  axis 2: log10(AR)                       N= 17   [-0.8000 .. 0.8000]   step≈0.1000
  axis 3: cos(theta_o) half-domain        N= 13   [0.0000 .. 1.0000]   step≈0.0833
  axis 4: phi_o [rad]                     N= 21   [0.0000 .. 3.1416]   step≈0.1571

Datasets in each wavelength group (example: wl_0p453)
  S_fw_phi_im                shape=(16, 21, 17, 13, 21)            dtype=float64
  S_fw_phi_re                shape=(16, 21, 17, 13, 21)            dtype=float64
  S_fw_theta_im              shape=(16, 21, 17, 13, 21)            dtype=float64
  S_fw_theta_re              shape=(16, 21, 17, 13, 21)            dtype=float64
  conv

In [3]:
"""Cell 2: Retrieve results by parameter values or indices.

Two ways to specify each axis:
  - By physical value: set *_target variables (nearest grid point is used).
  - By index: set *_index variables (overrides the corresponding target).
Set an index to None to use the target value instead.
"""

# ── Specify by physical value ──────────────────────────────────────
wl_0_target          = 0.453   # wavelength [um]
D_ve_target          = 0.3     # volume-equivalent diameter [um]
RI_real_target       = 1.66    # Re(m_p)
log_AR_target        = 0.85    # log10(AR), 0 = sphere
cos_theta_o_target   = 0.0     # cos(theta_o), half-domain [0, 1]
phi_o_target         = 1     # phi_o [rad], domain [0, pi]

# ── Or specify by index (set to None to use target value) ─────────
wl_0_index           = None    # int or None
D_ve_index           = None    # int or None
RI_real_index        = None    # int or None
log_AR_index         = None    # int or None
cos_theta_o_index    = None    # int or None
phi_o_index          = None    # int or None

# ── Resolve wavelength group ──────────────────────────────────────
if wl_0_index is not None:
    wl_names = sorted([n for n in f
                       if isinstance(f[n], h5py.Group) and "wl_0" in f[n].attrs])
    grp = f[wl_names[wl_0_index]]
else:
    grp = None
    for name in f:
        obj = f[name]
        if isinstance(obj, h5py.Group) and "wl_0" in obj.attrs:
            if abs(obj.attrs["wl_0"] - wl_0_target) < 1e-4:
                grp = obj
                break
    if grp is None:
        wl_list = [f[n].attrs["wl_0"] for n in f
                   if isinstance(f[n], h5py.Group) and "wl_0" in f[n].attrs]
        raise ValueError(f"Wavelength {wl_0_target} not found. Available: {wl_list}")

# ── Resolve grid indices ──────────────────────────────────────────
def _resolve(grid_key, target, index):
    v = f[grid_key][:]
    if index is not None:
        return int(index), v[index]
    i = int(np.argmin(np.abs(v - target)))
    return i, v[i]

i_dve, val_dve = _resolve("D_ve_grid",             D_ve_target,        D_ve_index)
i_ri,  val_ri  = _resolve("RI_real_grid",           RI_real_target,     RI_real_index)
i_ar,  val_ar  = _resolve("log_AR_grid",            log_AR_target,      log_AR_index)
i_u,   val_u   = _resolve("cos_theta_o_half_grid",  cos_theta_o_target, cos_theta_o_index)
i_ph,  val_ph  = _resolve("phi_o_grid",             phi_o_target,       phi_o_index)

# ── Display selected grid point ──────────────────────────────────
print("=" * 60)
print("Selected grid point")
print("=" * 60)
print(f"  wl_0         = {grp.attrs['wl_0']:.4f} um   (group: {grp.name})")
print(f"  D_ve         = {val_dve:.4f} um   [index {i_dve}]")
print(f"  RI_real      = {val_ri:.4f}       [index {i_ri}]")
print(f"  log10(AR)    = {val_ar:.4f}       [index {i_ar}]   → AR = {10**val_ar:.4f}")
print(f"  cos(theta_o) = {val_u:.4f}        [index {i_u}]")
print(f"  phi_o        = {val_ph:.4f} rad   [index {i_ph}]")
print(f"  converged    = {bool(grp['converged'][i_dve, i_ri, i_ar])}")

# ── Extract forward-scattering amplitude ─────────────────────────
idx = (i_dve, i_ri, i_ar, i_u, i_ph)
S_fw_theta = grp["S_fw_theta_re"][idx] + 1j * grp["S_fw_theta_im"][idx]
S_fw_phi   = grp["S_fw_phi_re"][idx]   + 1j * grp["S_fw_phi_im"][idx]

print("\n" + "=" * 60)
print("DDA results")
print("=" * 60)
print(f"  S_fw_theta = {S_fw_theta:.6e}")
print(f"  S_fw_phi   = {S_fw_phi:.6e}")

Selected grid point
  wl_0         = 0.4530 um   (group: /wl_0p453)
  D_ve         = 0.3000 um   [index 2]
  RI_real      = 1.6500       [index 18]
  log10(AR)    = 0.8000       [index 16]   → AR = 6.3096
  cos(theta_o) = 0.0000        [index 0]
  phi_o        = 0.9425 rad   [index 6]
  converged    = True

DDA results
  S_fw_theta = 4.215272e-01+2.120866e-01j
  S_fw_phi   = 1.093653e-01+2.185564e-01j
